# Libraries

In [1]:
!pip install -q torch==2.9.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip -q install "lm_eval[hf]"
!pip -q install compressed-tensors
!pip -q install optimum
!pip -q install gptqmodel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 201.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 209.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 145.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 93.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 101.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 158.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 127.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 102.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 MB 108.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 105.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 183.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━

In [2]:
# 2. Libraries
import random
import numpy as np
import torch
import os
import gc

import lm_eval
import subprocess

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

import shutil
from IPython.display import FileLink

In [3]:
import sys, transformers

print(sys.version)
print(torch.__version__)
print(transformers.__version__)
print(torch.cuda.is_available())

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
2.9.0+cu126
5.4.0
True


# Functions

In [4]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Global Variables

In [5]:
model_name = "EdgeCompress01/Llama-3.2-3B-Instruct-WANDA-GPTQ-4bit"
subfolder = "Models/25"
SEED = 42
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Seed for reproducibility

In [6]:
set_reproducibility(SEED)

Global seed set to 42


# Login to huggingface

In [7]:
# --- 3. HUGGING FACE AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Do Evaluation

**Configurations**

In [8]:
tasks_config = {
    "gsm8k": {
        "task_name": "gsm8k_cot",
        "fewshot": 2   # 🔻 reduced from 8
    },
    "mmlu": {
        "task_name": "mmlu",
        "fewshot": 2   # 🔻 reduced from 5
    },
    "arc_challenge": {
        "task_name": "arc_challenge",
        "fewshot": 0   # keep
    },
    "wikitext": {
        "task_name": "wikitext",
        "fewshot": 0   # keep
    }
}

In [9]:
for task, cfg in tasks_config.items():
    print(f"Starting evaluation for: {task}")

    model_args = f"pretrained={model_name},subfolder={subfolder},max_length=2048,dtype=float16,parallelize=True"
    command = [
        "lm_eval",
        "--model", "hf",
        "--model_args", model_args,
        "--tasks", cfg["task_name"],
        "--batch_size", str(3),   
        "--verbosity", "WARNING",
        "--seed", str(SEED),
        "--limit", str(100),
        "--num_fewshot", str(cfg["fewshot"]),
        "--output_path", f"evaluation_results/{task}"
    ]
    subprocess.run(command, check=True)
    
    torch.cuda.empty_cache()
    gc.collect()
    print(f"Finished {task}\n")

Starting evaluation for: gsm8k


2026-03-29:17:15:55 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-29:17:16:02 INFO     [_cli.run:376] Selected Tasks: ['gsm8k_cot']
2026-03-29:17:16:02 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-WANDA-GPTQ-4bit appears to be an instruct or chat variant but chat template is not applied.
        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu126 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attrib


{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
WARN  Feature `utils/Perplexity` requires Python < 3.14 and Python GIL enabled and Python >= 3.13.3T (T for Threading-Free edition of Python) plus Torch 2.8. Feature is currently skipped/disabled.
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-mon

[W329 17:16:16.802435522 Context.cpp:470] Warning: torch.backends.cuda.preferred_linalg_library is an experimental feature. If you see any error or unexpected behavior when this flag is set please file an issue on GitHub. (function operator())
INFO:httpx:HTTP Request: HEAD https://huggingface.co/EdgeCompress01/Llama-3.2-3B-Instruct-WANDA-GPTQ-4bit/resolve/main/Models/25/model.safetensors "HTTP/1.1 302 Found"


{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 5.8.0
Transformers : 5.4.0
Torch        : 2.9.0+cu126
Triton       : 3.5.0
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}


...

{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  HFKernelLinear: loaded CPU gemm_4bit kernel from `kernels-community/quantization-gptq` variant `torch29-cxx11-cpu-x86_64-linux`.
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  Kernel: Auto-selection: adding candidate `TritonV2QuantLinear`           
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  Kernel: selected -> `TritonV2QuantLinea

Loading weights: 100%|██████████| 842/842 [00:00<00:00, 1172.25it/s]
INFO:httpx:HTTP Request: HEAD https://huggingface.co/EdgeCompress01/Llama-3.2-3B-Instruct-WANDA-GPTQ-4bit/resolve/main/Models/25/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/EdgeCompress01/Llama-3.2-3B-Instruct-WANDA-GPTQ-4bit/c1709c7505e190c22986c4732cc970e4d315c6c4/Models%2F25%2Fgeneration_config.json "HTTP/1.1 200 OK"


{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  QuantizeConfig: offload_to_disk_path auto set to `./gptqmodel_offload/yinelchz-kldtsqdh/`
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  Format: Converting `checkpoint_format` from `FORMAT.GPTQ` to internal `FORMAT.GPTQ_V2`.
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  Format: Converting GPTQ v1 to v2                                

2026-03-29:17:42:46 WARNING  [api.task:729] [Task: wikitext] metric word_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-03-29:17:42:46 WARNING  [api.task:741] [Task: wikitext] metric word_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
2026-03-29:17:42:46 WARNING  [api.task:729] [Task: wikitext] metric byte_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-03-29:17:42:46 WARNING  [api.task:741] [Task: wikitext] metric byte_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
2026-03-29:17:42:46 WARNING  [api.task:729] [Task: wikitext] metric bits_per_byte is defined, but aggregation is not. using default aggregation=bits_per_byte
2026-03-29:17:42:46 WARNING  [api.task:741] [Task: wikitext] metric bits_per_byte is defined, but higher_is_better is not. using default higher_is_better=False
INFO:httpx:HTTP Request: H

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-WANDA-GPTQ-4bit', 'subfolder': 'Models/25', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 0, batch_size: 3
| Tasks  |Version|Filter|n-shot|    Metric     |   | Value |   |Stderr|
|--------|------:|------|-----:|---------------|---|------:|---|------|
|wikitext|      2|none  |     0|bits_per_byte  |↓  | 0.7478|±  |   N/A|
|        |       |none  |     0|byte_perplexity|↓  | 1.6792|±  |   N/A|
|        |       |none  |     0|word_perplexity|↓  |15.9853|±  |   N/A|

Finished wikitext



# Save reports

In [10]:
zip_path = shutil.make_archive(f"evaluation_results", 'zip', f"evaluation_results")
FileLink(zip_path)

/kaggle/working/evaluation_results.zip